In [ ]:
# SYSTEM CHECK (RAM)
import psutil

ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")

Total RAM: 230.05 GB
Available RAM: 227.29 GB


In [ ]:
# INSTALL REQUIREMENTS
!pip install opencv-python
!pip install tensorflow
!pip install openpyxl


In [ ]:
# IMPORT LIBRARIES
import zipfile
import os
import cv2
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import GRU, Dense, Dropout, TimeDistributed, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Nadam
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt


In [ ]:
# GLOBAL CONSTANTS
NUM_FRAMES = 120                 # Sequential frames per video
MIN_FRAMES = 120                # Minimum frames required
max_frames = 120                # Maximum frames extracted
IMAGE_HEIGHT, IMAGE_WIDTH = 224, 224

In [ ]:
# GOOGLE DRIVE SET
from google.colab import drive
drive.mount('drive')

zip_folder = 'drive/My Drive/Colab Notebooks_videoAnalytics/all_phs1_phs6lv1_lv38_vids.zip'
extract_path = ''

with zipfile.ZipFile(zip_folder, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete.")

video_count = len(os.listdir('all_phs1_phs6lv1_lv38_vids'))
print(f"Number of videos in the folder: {video_count}")

Mounted at drive
Extraction complete.
Number of videos in the folder: 1351


In [ ]:
# LOAD METADATA
metadata_file = pd.read_excel('all_phs1_phs6lv1_lv38_vids_meta_data.xlsx')
metadata_file
metadata_file.shape

(1351, 3)

In [ ]:
# FRAME PREPROCESSING
def preprocess_frame(frame, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT)):
    """
    Center crop, resize, and normalize a frame for model input.
    """
    h, w = frame.shape[:2]
    min_dim = min(h, w)
    start_x = (w - min_dim) // 2
    start_y = (h - min_dim) // 2
    cropped_frame = frame[start_y:start_y + min_dim, start_x:start_x + min_dim]

    frame_resized = cv2.resize(cropped_frame, target_size)

    # NORMALIZATION FOR Xception
    frame_resized = frame_resized.astype("float32")
    frame_resized = frame_resized / 127.5 - 1

    return frame_resized

In [ ]:
def extract_all_frames(video_path, max_frames):
    """
    Extracts up to `max_frames` sequential frames from a video.
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0

    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame = preprocess_frame(frame)
        frames.append(frame)
        frame_count += 1

    cap.release()

    frames_array = np.array(frames)
    print(f"Extracted {len(frames_array)} frames from {video_path}. Target: {max_frames}")
    return frames_array

In [ ]:
def load_video_data(data_path, metadata_file, num_frames=NUM_FRAMES):
    """
    Loads and processes videos into ML samples.
    """
    metadata = pd.read_excel(metadata_file)
    X, y = [], []
    skipped_videos = []

    for _, row in metadata.iterrows():
        video_name = row['video_name']
        video_path = os.path.join(data_path, video_name)

        try:
            all_frames = extract_all_frames(video_path, num_frames)

            if all_frames is None or len(all_frames) < num_frames:
                skipped_videos.append((video_name, "insufficient frames"))
                continue

            X.append(all_frames)

            label = 1 if row['tag'].strip().lower() == 'positive' else 0
            y.append(label)

        except Exception as e:
            skipped_videos.append((video_name, f"exception: {str(e)}"))

    print("\nSkipped videos summary:")
    for vid, reason in skipped_videos:
        print(f"- {vid}: {reason}")

    return np.array(X), np.array(y)

In [ ]:
# MAIN EXECUTION
if __name__ == "__main__":
    data_path = 'all_phs1_phs6lv1_lv38_vids'
    metadata_file = 'all_phs1_phs6lv1_lv38_vids_meta_data.xlsx'

    print("Loading and preprocessing video data...")
    X, y = load_video_data(data_path, metadata_file)

    print("Shape of X after loading and normalization:", X.shape)
    print("Shape of y after loading:", y.shape)

    print("Saving preprocessed arrays...")
    np.save('X_prep_120_frames.npy', X)
    np.save('y_labels_120_frames.npy', y)

    print("Feature shape:", X.shape)
    print("Label shape:", y.shape)


print("Features (X) shape:", X.shape)
print("Labels (y) shape:", y.shape)

Loading and preprocessing video data...
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_0°C,14-10-2024,02-32-12PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_1°C,14-10-2024,02-32-21PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_0°C,14-10-2024,02-32-29PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_0°C,14-10-2024,02-32-37PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_0°C,14-10-2024,02-32-44PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_1°C,14-10-2024,02-32-52PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,24_9°C,14-10-2024,02-33-00PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,25_0°C,14-10-2024,02-33-08PM.mp4. Target: 120
Extracted 120 frames from all_phs1_phs6lv1_lv38_vids/larvae_temp,24_8°C,

In [ ]:

# FORk REPRODUCIBILITY
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
from tensorflow.keras.utils import to_categorical


# TRAIN-VALIDATION SPLIT
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=SEED
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

X_train shape: (945, 120, 224, 224, 3)
X_val shape: (406, 120, 224, 224, 3)


In [ ]:
# ONE-HOT ENCODING
num_classes = 2
y_train_oh = to_categorical(y_train, num_classes=num_classes)
y_val_oh   = to_categorical(y_val, num_classes=num_classes)

In [ ]:
# FEATURE EXTRACTION (Xception)
base_model = Xception(
    weights="imagenet",
    include_top=False,
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3)
)

global_avg_layer = GlobalAveragePooling2D()
feature_extractor = Model(inputs=base_model.input, outputs=global_avg_layer(base_model.output))

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# GRU-BASED MODEL DEFINITION
from tensorflow.keras.layers import Input

input_layer = Input(shape=(NUM_FRAMES, IMAGE_HEIGHT, IMAGE_WIDTH, 3))

x_ception_td = TimeDistributed(feature_extractor)(input_layer)

gru_1 = GRU(1024, return_sequences=True)(x_ception_td)
dropout_1 = Dropout(0.5)(gru_1)

gru_2 = GRU(512)(dropout_1)
dropout_2 = Dropout(0.5)(gru_2)

dense_1 = Dense(128, activation="relu")(dropout_2)
dropout_3 = Dropout(0.5)(dense_1)

output_layer = Dense(num_classes, activation="sigmoid")(dropout_3)

model = Model(inputs=input_layer, outputs=output_layer)

In [ ]:
# COMPILE MODEL
optimizer = Nadam(learning_rate=1e-4)

model.compile(
    loss="categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 120, 224, 224,  │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 120, 2048)      │    20,861,480 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 120, 1024)      │     9,443,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 120, 1024)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 512)            │     2,362,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,733,098 (124.87 MB)

 Trainable params: 32,678,570 (124.66 MB)

 Non-trainable params: 54,528 (213.00 KB)

In [ ]:
# EARLY STOPPING
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
# CLASS WEIGHTS
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
w0 = class_weights[0]
w1 = class_weights[1]

In [ ]:
# CHUNKED TRAINING (MEMORY SAFE)
NUM_CHUNKS = 10

X_train_chunks = np.array_split(X_train, NUM_CHUNKS)
y_train_chunks = np.array_split(y_train_oh, NUM_CHUNKS)

print(f"Training data split into {NUM_CHUNKS} chunks.")

all_history = []

for i, (X_chunk, y_chunk) in enumerate(zip(X_train_chunks, y_train_chunks)):

    print(f"\n==============================")
    print(f"Training on chunk {i+1}/{NUM_CHUNKS}")
    print(f"Chunk shape: {X_chunk.shape}")
    print(f"==============================")

    history_chunk = model.fit(
        X_chunk,
        y_chunk,
        epochs=10,
        batch_size=3,
        class_weight={0: w0, 1: w1},
        callbacks=[early_stopping],
        validation_data=(X_val, y_val_oh),
        verbose=1
    )

    all_history.append(history_chunk.history)

    # MEMORY CLEANUP
    del X_chunk, y_chunk
    import gc
    gc.collect()

print("\nTraining completed across all chunks.")

Training data split into 10 chunks.

Training on chunk 1/10
Chunk shape: (95, 120, 224, 224, 3)
Epoch 1/10


### Compress `X_prep_120_frames.npy` to speed up download

In [ ]:
import numpy as np
import zipfile
import os

# Load the numpy array
X = np.load('X_prep_120_frames.npy')

# Save the numpy array to a compressed .npz file
# This will store it in a compressed format within a zip archive, which is more efficient for large arrays.
output_filename = 'X_prep_120_frames.npz'
np.savez_compressed(output_filename, X_prep_120_frames=X)

if os.path.exists(output_filename):
    print(f'Successfully compressed X_prep_120_frames.npy to {output_filename}')
else:
    print(f'Error: Failed to create {output_filename}. The file does not exist after compression.')

Successfully compressed X_prep_120_frames.npy to X_prep_120_frames.npz


In [ ]:
import shutil
import os

# Define the source files and destination folder
source_files = ['X_prep_120_frames.npz', 'y_labels_120_frames.npy']
destination_folder = 'drive/My Drive/pereprocessed_120_frames in numpy_array/' # You can specify a subfolder like 'drive/My Drive/Colab Notebooks/'

# Copy each file to Google Drive
for source_file in source_files:
    try:
        shutil.copy(source_file, destination_folder)
        print(f'Successfully copied {source_file} to {destination_folder}')
    except FileNotFoundError:
        print(f'Error: {source_file} not found. Please ensure it has been generated.')
    except Exception as e:
        print(f'Error copying {source_file}: {e}')

Successfully copied X_prep_120_frames.npz to drive/My Drive/pereprocessed_120_frames in numy_array/
Successfully copied y_labels_120_frames.npy to drive/My Drive/pereprocessed_120_frames in numy_array/


In [ ]:
#view file contents copied  to drive
!ls drive/My\ Drive/pereprocessed_120_frames\ in\ numpy_array/



X_prep_120_frames.npz  y_labels_120_frames.npy
